# Data Validation Plotly Graphs

Generates interactive HTML plots for each groundwater series.

- Groups multi-sensor CSVs (from `_sensor_N` suffixed files) into a single plot
- Validation flags: **Step change** (v2), **Constant head** (v3, 96-step runs within 15cm of min), **IQR** (v4)
- Subplot 1 (top): head series in dark colours with filled area + flag markers
- Subplot 2 (bottom): precipitation (blue), evapotranspiration (orange), recharge P-E (purple)
- Output: `output_data/wiertsema/<origin>/figures/<series>.html`

In [1]:
import sys
import io
import contextlib
import re
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from functions.validation_functions import (
    remove_duplicate_and_fill_missing,
    flag_unrealistic_step_change,
    flag_statistical_outliers,
)

dataset_roots = [
    repo_root / 'output_data' / 'wiertsema',
    repo_root / 'output_data' / 'fugro',
]
stressor_dir = repo_root / 'input_stressors'

MAX_UP = 0.3
MAX_DOWN = -0.05
BAND_UPPER_M = 0.15
CONST_STEPS = 96  # 96 hours = 4 days

for dr in dataset_roots:
    print(f'Dataset root: {dr}')
print(f'Stressor dir: {stressor_dir}')

Dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\wiertsema
Dataset root: d:\Users\jvanruitenbeek\data_validation\output_data\fugro
Stressor dir: d:\Users\jvanruitenbeek\data_validation\input_stressors


In [2]:
# Load precipitation and evapotranspiration from hourly KNMI file
knmi_hourly_path = stressor_dir / 'knmi_249_berkhout_hourly.csv'
print(f'KNMI hourly file: {knmi_hourly_path}')

df_knmi = pd.read_csv(knmi_hourly_path, index_col=0, parse_dates=True,
                      encoding='utf-8-sig', encoding_errors='replace')

prec_series = pd.to_numeric(df_knmi['precipitation_mm'], errors='coerce').dropna()
evap_series = pd.to_numeric(df_knmi['makkink_mm'], errors='coerce').dropna()
recharge_series = prec_series.subtract(evap_series, fill_value=0).dropna()

print(f'Precipitation: {len(prec_series)} hourly values')
print(f'Evaporation:   {len(evap_series)} hourly values')
print(f'Recharge:      {len(recharge_series)} hourly values')

KNMI hourly file: d:\Users\jvanruitenbeek\data_validation\input_stressors\knmi_249_berkhout_hourly.csv
Precipitation: 56760 hourly values
Evaporation:   56760 hourly values
Recharge:      56760 hourly values


In [3]:
HEAD_COLORS = [
    ('black',        'rgba(0, 0, 0, 0.12)'),
    ('#555555',      'rgba(85, 85, 85, 0.12)'),
    ('#8B4513',      'rgba(139, 69, 19, 0.12)'),
    ('#2F4F4F',      'rgba(47, 79, 79, 0.12)'),
    ('#6B4226',      'rgba(107, 66, 38, 0.12)'),
]


def get_series_base(stem):
    return re.sub(r'_sensor_\d+$', '', stem)


def group_sensor_files(csv_files):
    groups = {}
    for p in csv_files:
        base = get_series_base(p.stem)
        groups.setdefault(base, []).append(p)
    for base in groups:
        groups[base].sort()
    return groups


def flag_constant_head_runs(df_in, min_run_steps=96, band_upper_m=0.15, vcol='v3'):
    """Flag head values that stay within [hmin, hmin + band] for >= min_run_steps consecutive steps."""
    df = df_in.copy()
    if vcol not in df.columns:
        df[vcol] = np.nan

    valid = df['head'].dropna()
    if valid.empty:
        return df, False

    hmin = float(valid.min())
    threshold = hmin + float(band_upper_m)

    in_band = df['head'].notna() & (df['head'] >= hmin) & (df['head'] <= threshold)

    # Find runs of consecutive True values >= min_run_steps
    group_id = (~in_band).cumsum()
    run_lengths = in_band.groupby(group_id).transform('sum')
    mask_flagged = in_band & (run_lengths >= min_run_steps)

    df.loc[mask_flagged & df[vcol].isna(), vcol] = df.loc[mask_flagged, 'head']
    df.loc[mask_flagged, 'head'] = np.nan

    return df, mask_flagged.any()


def load_and_validate(csv_path):
    df = pd.read_csv(csv_path, index_col=0, parse_dates=True,
                     encoding='utf-8-sig', encoding_errors='replace')
    if 'head' not in df.columns:
        if df.shape[1] >= 1:
            df.columns = ['head'] + list(df.columns[1:])
        else:
            return None

    df.index = pd.to_datetime(df.index, errors='coerce')
    df = df[~df.index.isna()].sort_index()
    df.index.name = 'Time'

    with contextlib.redirect_stdout(io.StringIO()):
        df, _ = remove_duplicate_and_fill_missing(df)
        # head_raw is now set (complete original series)
        df, _ = flag_unrealistic_step_change(df, max_up=MAX_UP, max_down=MAX_DOWN)
        if 'dH' in df.columns:
            df = df.drop(columns=['dH'])
        df, _ = flag_constant_head_runs(df, min_run_steps=CONST_STEPS, band_upper_m=BAND_UPPER_M)
        df, _ = flag_statistical_outliers(df)

    return df


def create_series_plot(series_name, sensor_dfs, prec, evap, recharge):
    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.7, 0.3],
        shared_xaxes=True,
        vertical_spacing=0.06,
    )

    # ---- Row 1: Head traces per sensor ----
    for i, (label, df) in enumerate(sensor_dfs):
        line_color, fill_color = HEAD_COLORS[i % len(HEAD_COLORS)]
        raw_col = 'head_raw' if 'head_raw' in df.columns else 'head'

        # Original (complete) series
        fig.add_trace(go.Scatter(
            x=df.index, y=df[raw_col],
            mode='lines',
            name=label,
            line=dict(color=line_color, width=1.5),
            fill='tozeroy',
            fillcolor=fill_color,
        ), row=1, col=1)

        # Flag markers
        if 'v2' in df.columns:
            mask = df['v2'].notna()
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=df.index[mask], y=df.loc[mask, 'v2'],
                    mode='markers',
                    name=f'Step change ({label})',
                    marker=dict(color='red', size=6, symbol='circle-open'),
                ), row=1, col=1)

        if 'v3' in df.columns:
            mask = df['v3'].notna()
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=df.index[mask], y=df.loc[mask, 'v3'],
                    mode='markers',
                    name=f'Constant head ({label})',
                    marker=dict(color='tomato', size=6, symbol='diamond-open'),
                ), row=1, col=1)

        if 'v4' in df.columns:
            mask = df['v4'].notna()
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=df.index[mask], y=df.loc[mask, 'v4'],
                    mode='markers',
                    name=f'IQR ({label})',
                    marker=dict(color='firebrick', size=6, symbol='square-open'),
                ), row=1, col=1)

    # Time range from head data
    all_times = pd.DatetimeIndex([])
    for _, df in sensor_dfs:
        all_times = all_times.union(df.index)
    if len(all_times) == 0:
        return fig
    t_min, t_max = all_times.min(), all_times.max()

    # Clamp head y-axis to actual data range
    raw_parts = []
    for _, df in sensor_dfs:
        col = 'head_raw' if 'head_raw' in df.columns else 'head'
        raw_parts.append(df[col].dropna())
    for vcol in ('v2', 'v3', 'v4'):
        for _, df in sensor_dfs:
            if vcol in df.columns:
                raw_parts.append(df[vcol].dropna())
    all_vals = pd.concat(raw_parts)
    if not all_vals.empty:
        y_min, y_max = float(all_vals.min()), float(all_vals.max())
        pad = (y_max - y_min) * 0.08 if y_max > y_min else 0.5
        fig.update_yaxes(range=[y_min - pad, y_max + pad], row=1, col=1)

    # ---- Row 2: Forcings subplot ----
    p = prec[(prec.index >= t_min) & (prec.index <= t_max)]
    e = evap[(evap.index >= t_min) & (evap.index <= t_max)]
    r = recharge[(recharge.index >= t_min) & (recharge.index <= t_max)]

    if len(p) > 0:
        fig.add_trace(go.Scatter(
            x=p.index, y=p.values,
            name='Precipitation',
            fill='tozeroy',
            line=dict(color='steelblue', width=0.5),
            fillcolor='rgba(30, 100, 200, 0.4)',
        ), row=2, col=1)

    if len(e) > 0:
        fig.add_trace(go.Scatter(
            x=e.index, y=-e.values,
            name='Evapotranspiration',
            fill='tozeroy',
            line=dict(color='darkorange', width=0.5),
            fillcolor='rgba(255, 140, 0, 0.4)',
        ), row=2, col=1)

    if len(r) > 0:
        fig.add_trace(go.Scatter(
            x=r.index, y=r.values,
            mode='lines',
            name='Recharge (P-E)',
            line=dict(color='purple', width=1.5),
        ), row=2, col=1)

    fig.update_layout(
        title=dict(text=series_name, x=0.5),
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', y=1.10, x=0.5, xanchor='center'),
        height=750,
        margin=dict(t=100),
    )
    fig.update_yaxes(title_text='Head [m NAP]', row=1, col=1)
    fig.update_yaxes(title_text='P / E / Recharge [mm]', row=2, col=1)
    fig.update_xaxes(title_text='Time', row=2, col=1)

    return fig

In [4]:
total_plots = 0

for dataset_root in dataset_roots:
    if not dataset_root.exists():
        print(f'\nSkipping {dataset_root} (does not exist)')
        continue

    all_origins = sorted([d for d in dataset_root.iterdir() if d.is_dir()])
    dataset_label = dataset_root.name  # 'wiertsema' or 'fugro'

    print(f'\n{"#"*60}')
    print(f'  Dataset: {dataset_label}')
    print(f'{"#"*60}')

    for origin_dir in all_origins:
        csv_dir = origin_dir / 'only_csv'
        if not csv_dir.exists():
            continue

        csv_files = sorted(csv_dir.glob('*.csv'))
        if not csv_files:
            continue

        fig_dir = origin_dir / 'figures'
        fig_dir.mkdir(parents=True, exist_ok=True)

        groups = group_sensor_files(csv_files)
        origin_name = origin_dir.name
        print(f'\n{"="*60}')
        print(f'Origin: {origin_name} ({len(groups)} series, {len(csv_files)} files)')
        print(f'{"="*60}')

        for base_name, file_list in sorted(groups.items()):
            sensor_dfs = []
            for f in file_list:
                match = re.search(r'_sensor_(\d+)$', f.stem)
                label = f'Sensor {match.group(1)}' if match else 'Head'

                df = load_and_validate(f)
                if df is None:
                    continue
                raw_col = 'head_raw' if 'head_raw' in df.columns else 'head'
                if df[raw_col].notna().any():
                    sensor_dfs.append((label, df))

            if not sensor_dfs:
                print(f'  {base_name}: no valid data, skipping')
                continue

            fig = create_series_plot(base_name, sensor_dfs,
                                    prec_series, evap_series, recharge_series)

            html_path = fig_dir / f'{base_name}.html'
            fig.write_html(str(html_path))
            total_plots += 1

            n_sensors = len(sensor_dfs)
            print(f'  {base_name}: {n_sensors} sensor(s) -> {html_path.name}')

print(f'\n{"="*60}')
print(f'Done. Generated {total_plots} HTML plots.')


############################################################
  Dataset: wiertsema
############################################################

Origin: Beemster_86349_1_deel_1 (35 series, 37 files)
  86349-1 HB029PB01 HB_BE0213+10_BIB_GMW_PB1_F-599: 1 sensor(s) -> 86349-1 HB029PB01 HB_BE0213+10_BIB_GMW_PB1_F-599.html
  86349-1 HB030PB01 HB_BE0213+10_BIT_GMW_PB1_F-681: 1 sensor(s) -> 86349-1 HB030PB01 HB_BE0213+10_BIT_GMW_PB1_F-681.html
  86349-1 HB031PB01 HB_BE0235+10_BIKR_GMW_PB1_F-235: 1 sensor(s) -> 86349-1 HB031PB01 HB_BE0235+10_BIKR_GMW_PB1_F-235.html
  86349-1 HB033PB01 HB_BE0235+10_BIT_GMW_PB1_F-648: 1 sensor(s) -> 86349-1 HB033PB01 HB_BE0235+10_BIT_GMW_PB1_F-648.html
  86349-1 HB034PB01 HB_BE0242+8_BUKR_GMW_PB1_F-250: 1 sensor(s) -> 86349-1 HB034PB01 HB_BE0242+8_BUKR_GMW_PB1_F-250.html
  86349-1 HB036PB01 HB_BE0242+7_BIT_GMW_PB1_F-654: 1 sensor(s) -> 86349-1 HB036PB01 HB_BE0242+7_BIT_GMW_PB1_F-654.html
  86349-1 HB037PB01 HB_BE0254+96_BUKR_GMW_PB1_F-198: 1 sensor(s) -> 86349-1